# Kafka Demo and Lab

Producer / topic / consumer with the `confluent_kafka` python library and a Kafka broker running in Docker
(chapter `04.kafka_overview` of the course).

## Instructions

- Part 1 is the demo of the course: the `timer` topic, the producer that sends the clock every second and the consumer that prints it.
- Part 2 is the lab: a new topic, a book from Project Gutenberg sent line by line, and a consumer that cleans the text (same cleaning as the word count lab) and writes it to a new file.
- The broker runs in a Docker container on `localhost:9092`, the notebook is the producer and the consumer.
- Note: in the course the producer and the consumer are two scripts in two terminals. In one notebook we only have one process, so we produce first and we consume after: Kafka keeps the messages in the topic, the consumer reads them from the beginning (`auto.offset.reset`), so the result is the same.

In [ ]:
# confluent_kafka is not preinstalled (skip this cell if it is already in the environment)
!pip install -q confluent-kafka

## Start the Kafka broker

The image `apache/kafka-native:4.1.1` is a Kafka in KRaft mode (no Zookeeper), it starts in a few seconds.
The command of the course is `docker run -p 9092:9092 apache/kafka-native:4.1.1`: it blocks the terminal, so
here we start it detached (`-d`) with a name, and the cell can be re-run without error.

In [19]:
import subprocess

CONTAINER = "kafka_lab"
IMAGE = "apache/kafka-native:4.1.1"

def sh(cmd):
    # small helper: run a shell command and return stdout (stderr is kept for the message)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).strip()

# is the container already running ? (the cell must be idempotent)
_, running = sh(f'docker ps --filter name={CONTAINER} --format "{{{{.Names}}}}"')

if CONTAINER in running:
    print(f"container {CONTAINER} already running")
else:
    rc, out = sh(f"docker run -d --rm --name {CONTAINER} -p 9092:9092 {IMAGE}")
    print("started" if rc == 0 else "error", out)

container kafka_lab already running


In [20]:
import time
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092"
config = {"bootstrap.servers": BOOTSTRAP}
admin_client = AdminClient(config)

# the broker needs a few seconds before it answers: we wait for the metadata instead of a blind sleep
for attempt in range(30):
    try:
        meta = admin_client.list_topics(timeout=2)
        print(f"broker ready after {attempt}s, cluster id = {meta.cluster_id}")
        break
    except Exception as e:
        time.sleep(1)
else:
    raise RuntimeError("broker not reachable on localhost:9092, check the docker container")

broker ready after 0s, cluster id = 5L6g3nShT-eMCtK--X86sw


## Part 1 - Demo

The three files `demo_py/admin.py`, `demo_py/producer.py` and `demo_py/consumer.py` of the course are repeated
here in three cells. The code and the comments of the course are kept as they are, we only changed three
things: the `BOOTSTRAP` variable, the streaming window shortened to 15 s, and the check of the future returned
by `create_topics`.

> **Demo 1.** `admin.py` : create the topic and list the topics of the cluster

In [21]:
# --- admin.py of the course ---
topic='timer'

# the course calls create_topics and moves on. But create_topics is asynchronous: it
# returns a dict {topic: Future}, so if the creation fails the error stays silent and
# the topic looks created. We ask the future for its result to really know, and we
# catch TOPIC_ALREADY_EXISTS which is what we get every time the cell is re-run.
for t, fut in admin_client.create_topics(
        [NewTopic(topic, num_partitions=1, replication_factor=1)]).items():
    try:
        fut.result()
        print(f"topic {t} created")
    except Exception as e:
        print(f"topic {t}: {e}")

x = admin_client.list_topics()
for  t in x.topics.keys():
  print(t)

topic timer: KafkaError{code=TOPIC_ALREADY_EXISTS,val=36,str="Topic 'timer' already exists."}
timer
__consumer_offsets
book_lines


`create_topics` gives back a future and not the actual topic, because the admin client sends the request to the
controller of the cluster and doesn't wait to any response. If we don't call `.result()` we never see the error that can occure, the topic looks created but it's not. When we re-run the cell we get `TOPIC_ALREADY_EXISTS` and that's normal because we catch it(the error).

In the list we only see `timer`. The internal topic `__consumer_offsets` is not there because Kafka creates it
the first time a consumer group commits an offset, so it will appear after we run the conssumer. It's the topic
where Kafka itself store the position of each group so it's a topic that Kafka uses to remember where we stopped
reading.

> **Demo 2.** `producer.py` : send the time every second during a small window

In [22]:
# --- producer.py of the course ---
import socket
from datetime import datetime, timedelta
from confluent_kafka import Producer

conf = {'bootstrap.servers': BOOTSTRAP,
        'client.id': socket.gethostname()}

producer = Producer(conf)

topic='timer'
message='The time is now '

duration = 5 # Streaming window in minutes
start_time = datetime.now() # Current clock time
stop_time = start_time + timedelta(minutes=duration) #start+duration=stop

# 5 minutes is long to wait in a notebook and the mechanism is already visible after a
# few messages, so we shorten the streaming window to 15 seconds.
stop_time = start_time + timedelta(seconds=15)

while datetime.now() < stop_time:
  time_now = datetime.now().strftime("%H:%M:%S")
  producer.produce(
    topic=topic,
    value=message + time_now
  )
  producer.poll(0)   # serves the delivery callbacks without blocking the loop
  print(message + time_now)
  time.sleep(1)

producer.flush()
producer.close()

The time is now 17:37:11


The time is now 17:37:12
The time is now 17:37:13
The time is now 17:37:14
The time is now 17:37:15
The time is now 17:37:16
The time is now 17:37:17
The time is now 17:37:18
The time is now 17:37:19
The time is now 17:37:20
The time is now 17:37:21
The time is now 17:37:22
The time is now 17:37:23
The time is now 17:37:24
The time is now 17:37:25


True

> **Demo 3.** `consumer.py` : read the topic and stop after 10 polls without message

In [23]:
# --- consumer.py of the course ---
from confluent_kafka import Consumer

conf = {'bootstrap.servers': BOOTSTRAP,
        'group.id': 'foo',
        'auto.offset.reset': 'smallest'}   # smallest = earliest, read from the start of the topic

consumer = Consumer(conf)

topic='timer'
consumer.subscribe([topic])

# Configuration
MAX_EMPTY_POLLS = 10  # Ends after ~10 seconds of silence
MAX_ERRORS = 5        # Ends after 5 consecutive errors
empty_polls = 0
error_count = 0

while True:
    msg = consumer.poll(1.0)

    # 1. Handle "No Message" (Timeout)
    if msg is None:
        empty_polls += 1
        if empty_polls >= MAX_EMPTY_POLLS:
            print("Closing: No new messages received.")
            break
        continue

    # 2. Handle Errors
    if msg.error():
        error_count += 1
        print(f"Consumer error: {msg.error()}")
        if error_count >= MAX_ERRORS:
            print("Closing: Too many consecutive errors.")
            break
        continue

    # 3. Handle Success
    # Reset counters when we actually get data
    empty_polls = 0
    error_count = 0

    print(f"Rcvd message: {msg.value().decode('utf-8')}")

# Clean up
consumer.close()

Rcvd message: The time is now 17:37:11
Rcvd message: The time is now 17:37:12
Rcvd message: The time is now 17:37:13
Rcvd message: The time is now 17:37:14
Rcvd message: The time is now 17:37:15
Rcvd message: The time is now 17:37:16
Rcvd message: The time is now 17:37:17
Rcvd message: The time is now 17:37:18
Rcvd message: The time is now 17:37:19
Rcvd message: The time is now 17:37:20
Rcvd message: The time is now 17:37:21
Rcvd message: The time is now 17:37:22
Rcvd message: The time is now 17:37:23
Rcvd message: The time is now 17:37:24
Rcvd message: The time is now 17:37:25
Closing: No new messages received.


The consumer received exactly the 15 messages of the producer, in the right order from 17:37:11 to 17:37:25,
and then it printed `Closing: No new messages received.` after 10 s of silence because no news frome the producer.

The consumer is a `while True` with a `poll(1.0)` and not a `for` on a list, a topic has no end because it's
a stream, the consumer asks "is there something new ?" every second. The two counters are just a way to stop
the loop in a notebook, in production the consumer never stops.

`poll` returns three different things and that's why there are three different `if` :first it can be `None` (nothing arrived during the
timeout), then it can be a message with `.error()` (a problem, for example the broker restarted), or fianally  a real message. 


## Part 2 - Lab

1. Examine the three .py files as examples. *(done in part 1)*
2. Create a new topic by adjusting or creating a new admin.py file.
3. Download any book from Project Gutenberg.
4. Create a new producer.py that will read the downloaded book line by line and send each line to the topic created in step 2.
5. Adjust the consumer.py so that it reads from the new topic created in step 2.
6. Using the text cleaning steps from the word count lab as inspiration, do some basic text cleaning and parsing for the incoming messages and output them to a new file.

> **Q2.** Create a new topic

In [24]:
BOOK_TOPIC = "book_lines"

def wait_topic(name, present, tries=30):
    """Waits until the topic is there (present=True) or gone (present=False). Creation and
    deletion are both asynchronous, so list_topics can still show the old state for a moment."""
    for _ in range(tries):
        if (name in admin_client.list_topics(timeout=5).topics) == present:
            return True
        time.sleep(1)
    return False

# the cell must be re-runnable: if the topic is already there we delete it first, otherwise the
# producer below adds the book a second time and every count is doubled
if BOOK_TOPIC in admin_client.list_topics(timeout=10).topics:
    for t, fut in admin_client.delete_topics([BOOK_TOPIC]).items():
        try:
            fut.result()
        except Exception as e:
            print(f"delete {t}: {e}")
    print(f"topic {BOOK_TOPIC} deleted:", wait_topic(BOOK_TOPIC, present=False))

# 3 partitions this time, replication 1 because we only have one broker
for t, fut in admin_client.create_topics(
        [NewTopic(BOOK_TOPIC, num_partitions=3, replication_factor=1)]).items():
    try:
        fut.result()
        print(f"topic {t} created with 3 partitions")
    except Exception as e:
        print(f"topic {t}: {e}")

if not wait_topic(BOOK_TOPIC, present=True):
    raise RuntimeError(f"{BOOK_TOPIC} still not in the metadata, re-run the cell")

t_meta = admin_client.list_topics(timeout=10).topics[BOOK_TOPIC]
print("partitions:", sorted(t_meta.partitions.keys()))

topic book_lines deleted: True
topic book_lines created with 3 partitions
partitions: [0, 1, 2]


3 partitions because the partition is the unit of parallelism ! One partition can only be read by one
consumer of a group, so with 1 partition a second consumer would do nothing. The price to pay is that the
order is only guaranteed inside a partition, so the lines will arrive mixed for the order but we don't care because this task is to count word ! 

> **Q3.** Download a book from Project Gutenberg

Any book was allowed, we take the same one as the lab 1 (*Around the World in 80 Days*, Gutenberg #103) so we
can compare the two results at the end.

In [25]:
import os
import requests

# as seen in the lab 1: same book (Gutenberg #103)
book_url = "https://www.gutenberg.org/ebooks/103.txt.utf-8"
book_file = "around_the_world_in_80_days.txt"

if not os.path.exists(book_file):
    response = requests.get(book_url)
    if response.status_code == 200:
        with open(book_file, "wb") as f:
            f.write(response.content)
        print(f"downloaded {book_file} ({len(response.content):,} bytes)")
    else:
        print("download failed:", response.status_code)
else:
    print(f"{book_file} already there")

# read in python and not with sc.textFile: the encoding has to be explicit on windows
with open(book_file, encoding="utf-8") as f:
    book_lines = f.read().splitlines()
print(f"{len(book_lines):,} lines")
print(book_lines[:5])

around_the_world_in_80_days.txt already there
8,312 lines
['The Project Gutenberg eBook of Around the World in Eighty Days', '    ', 'This eBook is for the use of anyone anywhere in the United States and', 'most other parts of the world at no cost and with almost no restrictions', 'whatsoever. You may copy it, give it away or re-use it under the terms']


> **Q4.** A producer that reads the book line by line and sends each line to the topic

In [26]:
delivered = 0
failed = 0

def delivery_report(err, msg):
    # the only way to know the broker really acked the message: produce() is asynchronous
    global delivered, failed
    if err is not None:
        failed += 1
        print("delivery failed:", err)
    else:
        delivered += 1

book_producer = Producer({"bootstrap.servers": BOOTSTRAP,
                          "client.id": socket.gethostname(),
                          "linger.ms": 50})      # wait 50 ms to group the lines in bigger batches

t0 = time.perf_counter()
for i, line in enumerate(book_lines):
    if not line.strip():
        continue                                 # skip the empty lines
    try:
        book_producer.produce(
            topic=BOOK_TOPIC,
            key=str(i).encode("utf-8"),          # the key chooses the partition (hash of the key)
            value=line.encode("utf-8"),          # kafka only transports bytes
            callback=delivery_report,
        )
    except BufferError:
        # local queue full: poll to empty it and retry the line
        book_producer.poll(1)
        book_producer.produce(topic=BOOK_TOPIC, key=str(i).encode("utf-8"),
                              value=line.encode("utf-8"), callback=delivery_report)
    book_producer.poll(0)                        # serve the callbacks

book_producer.flush()                            # blocks until the queue is empty
book_producer.close()

print(f"delivered: {delivered:,}   failed: {failed}   in {time.perf_counter() - t0:.1f} s")

delivered: 6,428   failed: 0   in 0.1 s


8 312 lines give 6 428 messages (we skip the empty lines), 0 failed, in 0.1 s. It is this fast because
`produce()` has nothing to send, it only puts the message in a local queue and a background thread sends the batches.

We have the hash part which is the `key` it's the line number and it chooses the partition (`hash(key) % 3`): 2 131 / 2 140 / 2 157, the 3
partitions are almost equal which is perfect.

> **Q5 & Q6.** A consumer on the new topic, that cleans the lines and writes them to a new file

**As seen in the lab 1**: lower case, split on the spaces and the apostrophes, remove the punctuation, remove
the stop words. Step 6 asks to reuse it, so we copy `PUNCT`, `STOPWORDS`, `clean()` and `tokenize()` and we
only add the function that works on one message.

In [27]:
# As seen in the lab 1: same PUNCT, same STOPWORDS, same clean(), same tokenize().
# Only clean_line() is new: the lab 1 chained the 3 steps in the rdd pipeline, here
# we need one function that takes one message.
import re, string
from collections import Counter

STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'at', 'by', 'for',
    'with', 'from', 'as', 'is', 'was', 'were', 'be', 'been', 'are', 'am', 'it', 'its',
    'he', 'she', 'they', 'them', 'his', 'her', 'their', 'him', 'i', 'you', 'we', 'me',
    'my', 'your', 'our', 'this', 'that', 'these', 'those', 'there', 'here', 'not', 'no',
    'so', 'if', 'then', 'than', 'which', 'who', 'whom', 'what', 'when', 'where', 'had',
    'has', 'have', 'do', 'did', 'does', 'would', 'could', 'should', 'will', 'shall',
    'may', 'might', 'can', 'up', 'out', 'into', 'over', 'all', 'any', 'some', 'very',
    'more', 'most', 'one', 'two', 'said', 'mr', 'mrs', 'himself', 'only', 'now', 'must',
    'about', 'after', 'before', 'through', 'upon', 'off', 'again', 'own', 'such',
    'other', 'each', 'while', 'being', 'made', 'like', 'much', 'well', 'go', 'went',
    # Project Gutenberg header and license, present in the file
    'project', 'gutenberg', 'ebook', 'www', 'org',
    # leftovers of contractions once the apostrophe is split: Fogg's, don't, I'll...
    's', 't', 'll', 're', 've', 'd', 'm', '',
}

# ASCII punctuation + typographic quotes and dashes found in the Gutenberg text
PUNCT = string.punctuation + '“”‘’—–…'
PUNCT_RE = re.compile('[' + re.escape(PUNCT) + ']')

def clean(word):
    return PUNCT_RE.sub('', word).strip()

def tokenize(line):
    # lower case + split on whitespace AND apostrophes (qu'il -> qu, il ; Fogg's -> fogg, s)
    return re.split(r"[\s'’]+", line.lower())

def clean_line(line):
    """One incoming message -> the list of its useful words."""
    return [w for w in (clean(t) for t in tokenize(line)) if w != '' and w not in STOPWORDS]

# quick check before plugging it on the stream
print(clean_line("Mr. Phileas Fogg lived, in 1872, at No. 7, Saville Row, Burlington Gardens."))

['phileas', 'fogg', 'lived', '1872', '7', 'saville', 'row', 'burlington', 'gardens']


In [28]:
import uuid

OUT_FILE = "cleaned_lines.txt"

conf = {"bootstrap.servers": BOOTSTRAP,
        # a new group.id at each run, otherwise the second run reads nothing (see the demo)
        "group.id": f"book_cleaner_{uuid.uuid4().hex[:8]}",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": True}

book_consumer = Consumer(conf)
book_consumer.subscribe([BOOK_TOPIC])

MAX_EMPTY_POLLS = 10
MAX_ERRORS = 5
empty_polls = 0
error_count = 0
n_msg = 0
n_words = 0
word_counts = Counter()      # replaces the reduceByKey of the lab 1
per_partition = Counter()

t0 = time.perf_counter()
with open(OUT_FILE, "w", encoding="utf-8") as out:
    while True:
        msg = book_consumer.poll(1.0)

        if msg is None:
            empty_polls += 1
            if empty_polls >= MAX_EMPTY_POLLS:
                print("Closing: No new messages received.")
                break
            continue

        if msg.error():
            error_count += 1
            print(f"Consumer error: {msg.error()}")
            if error_count >= MAX_ERRORS:
                print("Closing: Too many consecutive errors.")
                break
            continue

        empty_polls = 0
        error_count = 0

        # as seen in the lab 1, but message by message as the lines arrive, not on the whole file
        words = clean_line(msg.value().decode("utf-8"))
        per_partition[msg.partition()] += 1
        n_msg += 1
        if words:
            out.write(" ".join(words) + "\n")
            word_counts.update(words)
            n_words += len(words)

book_consumer.close()

print(f"{n_msg:,} messages consumed, {n_words:,} words kept, "
      f"{len(word_counts):,} different words, in {time.perf_counter() - t0:.1f} s")
print("messages per partition:", dict(sorted(per_partition.items())))
print("output file:", OUT_FILE)

Closing: No new messages received.
6,428 messages consumed, 32,751 words kept, 7,304 different words, in 14.0 s
messages per partition: {0: 2131, 1: 2140, 2: 2157}
output file: cleaned_lines.txt


In [29]:
# top 20, to compare with the lab 1
with open(OUT_FILE, encoding="utf-8") as f:
    for line in [next(f) for _ in range(5)]:
        print(line.rstrip())

print("\ntop 20 words:")
for w, c in word_counts.most_common(20):
    print(f"{w:<14}{c:>6}")

around world eighty days
using
around world eighty days
chapter phileas fogg passepartout accept master man
chapter ii passepartout convinced last found ideal

top 20 words:
fogg             645
passepartout     422
fix              256
phileas          255
aouda            136
master           128
time             125
train            119
sir              100
hundred           98
replied           93
steamer           91
hours             89
thousand          88
without           88
day               85
days              84
man               77
going             76
left              76


**As seen in the lab 1, the result is exactly the same result that we obtain**: 32 751 words kept, 7 304 different words,
same top 20 (fogg 645, passepartout 422, fix 256...). The result doesn't change, only the way to get it changes because in the lab 1, it
had the whole file and did one `reduceByKey` (batch), here the words arrive one line at a time and we keep a
`Counter` that we update at each message (streaming) which is the big difference between kafka and spark. 

The 14.6 s of the consumer against the 0.1 s of the producer is misleading: 10 of those seconds are the 10
`poll(1.0)` of silence we wait before stopping the loop. The real reading is around 3-4 s if we "throw away" the poll ! 
So to conclude same joke as in the lab 1 the French language is way more elaborate than the english one because england like to be straight to the point !

## Cleanup

The topics are deleted and the container stopped, so the next run starts from a clean cluster.
Uncomment to really do it.

In [30]:
# admin_client.delete_topics([topic, BOOK_TOPIC])
# sh(f"docker stop {CONTAINER}")   # --rm was used, so stopping also removes the container